In [ ]:
from google.cloud import bigquery
import geopandas as gpd

PROJECT = "sumai-search-20261031"  # GCPのプロジェクトID

# 完成品の2つの表を駅で結合して取ってくる
sql = """
select s.station_name, s.city_name, s.n_lines, c.nearest_terminal, c.commute_min_v0,
       st_astext(s.geom) as wkt
from marts.dim_stations s
join marts.station_commute_v0 c using (group_code)
"""
df = bigquery.Client(project=PROJECT).query(sql).to_dataframe()   # 結果を表として受け取る
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df.wkt), crs=4326)  # 地図データに戻す
print(len(gdf), "駅")
# 通勤時間で色分けした地図を表示（マウスを乗せると駅名などが出る）
gdf.explore(column="commute_min_v0", cmap="viridis_r",
            tooltip=["station_name", "city_name", "nearest_terminal", "commute_min_v0"])


In [ ]:
import webbrowser
from pathlib import Path
import folium

# ① 背景だけの地図を先に作る（地理院の淡色地図を半透明にする）
m = folium.Map(location=[35.68, 139.76], zoom_start=9, tiles=None)
folium.TileLayer(
    tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png",
    attr='<a href="https://maps.gsi.go.jp/development/ichiran.html">地理院タイル</a>',
    name="地理院タイル（淡色）",
    opacity=0.4,          # 0〜1。小さいほど背景が薄くなる
).add_to(m)

# ② その上に駅を重ねる（m=m で①の地図に描く）
gdf.explore(
    m=m,
    column="commute_min_v0",
    scheme="UserDefined",
    classification_kwds={"bins": [30, 45, 60, 75, 90, 120]},
    cmap="RdYlGn_r",
    marker_type="circle_marker",
    marker_kwds={"radius": 5},
    style_kwds={"color": "black", "weight": 0.7, "fillOpacity": 0.9},
    tooltip=["station_name", "city_name", "nearest_terminal", "commute_min_v0"],
)

out = Path("stations_map.html").resolve()
m.save(out)
webbrowser.open(out.as_uri())
